In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import KFold, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error

In [2]:
#使わなくなったやつ
#import pickle
#import lightgbm as lgb
#import xgboost as xgb

#from sklearn.model_selection import train_test_split
#from sklearn.model_selection import KFold, cross_val_score
#from catboost import CatBoostRegressor

In [3]:
#前処理したデータ
client_data = pd.read_csv("client_data.csv")
pre_data = pd.read_csv("pre_data.csv")

In [4]:
#Xとyに分割
client_X = client_data.drop(columns=["y"])
client_y = client_data["y"]

X = pre_data.drop(columns=["y"])
y = pre_data["y"]

In [5]:
# KFoldの設定
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
# パラメータの設定
param_grid = {
    'n_estimators': [50, 100, 500],
    'learning_rate': [0.001, 0.01, 0.1],
    'max_depth': [3, 4, 5]
}

In [7]:
# GridSearchCVの設定
client_grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)

# GridSearchCVの実行
client_grid_search.fit(client_X, client_y)

# ベストパラメータとスコアの表示
print(f'Best parameters: {client_grid_search.best_params_}')
print(f'Best MAE: {-client_grid_search.best_score_}')

Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500}
Best MAE: 0.0059081975618998205


In [8]:
# GridSearchCVの設定
grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)

# GridSearchCVの実行
grid_search.fit(X, y)

# ベストパラメータとスコアの表示
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best MAE: {-grid_search.best_score_}')

Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500}
Best MAE: 0.8331253435402303


In [9]:
# ベストモデルでKFoldを使って予測
client_best_model = client_grid_search.best_estimator_
client_kf_predictions = np.zeros(len(client_y))

best_model = grid_search.best_estimator_
kf_predictions = np.zeros(len(y))

In [10]:
# クロスバリデーションでモデルを学習し、予測値を計算(クライアント)
for client_train_index, client_val_index in kf.split(client_X):
    client_X_train, client_X_val = client_X.iloc[client_train_index], client_X.iloc[client_val_index]
    client_y_train, client_y_val = client_y.iloc[client_train_index], client_y.iloc[client_val_index]
    
    # ベストモデルで学習
    client_best_model.fit(client_X_train, client_y_train)
    
    # テストデータに対する予測
    client_y_pred = client_best_model.predict(client_X_test)
    
    # 全体の予測に対する平均を計算するために各Foldの予測を蓄積
    client_kf_predictions[client_test_index] = client_y_pred

NameError: name 'client_X_test' is not defined

In [ ]:
# クロスバリデーションでモデルを学習し、予測値を計算
for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    # ベストモデルで学習
    best_model.fit(X_train_fold, y_train)
    
    # テストデータに対する予測
    y_pred = best_model.predict(X_test)
    
    # 全体の予測に対する平均を計算するために各Foldの予測を蓄積
    kf_predictions[test_index] = y_pred

In [ ]:
# 各Foldの予測値の平均を最終予測値として計算
client_final_predictions = client_kf_predictions
final_predictions = kf_predictions

In [ ]:
# モデルの性能評価(クライアント)
client_mae = mean_absolute_error(client_y, client_final_predictions)
print(f'Mean Absolute Error(client): {client_mae}')

In [ ]:
# モデルの性能評価
mae = mean_absolute_error(y, final_predictions)
print(f'Mean Absolute Error: {mae}')

In [ ]:
# モデルを保存
joblib.dump(client_models, "Apple_client_model.pkl")
joblib.dump(models, "Apple_model.pkl")

In [ ]:
# 学習曲線のプロット
def plot_learning_curve(estimator, title, X, y, cv, scoring='neg_mean_absolute_error'):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv, scoring=scoring, n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 10)
    )

    train_scores_mean = -np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    val_scores_mean = -np.mean(val_scores, axis=1)
    val_scores_std = np.std(val_scores, axis=1)

    plt.figure()
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Mean Absolute Error")
    plt.ylim(0, 10)
    plt.grid()

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, val_scores_mean - val_scores_std,
                     val_scores_mean + val_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training error")
    plt.plot(train_sizes, val_scores_mean, 'o-', color="g", label="Cross-validation error")

    plt.legend(loc="best")
    return plt

In [ ]:
# XGBRegressorの学習曲線のプロット
plot_learning_curve(XGBRegressor(random_state=42), "Learning Curve (client)", client_X, client_y, cv=kf)
plt.show()

In [ ]:
# XGBRegressorの学習曲線のプロット
plot_learning_curve(XGBRegressor(random_state=42), "Learning Curve", X, y, cv=kf)
plt.show()